In [194]:
import pandas as pd
import numpy as np

In [195]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)

In [196]:
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
target = ['fuel_efficiency_mpg']

In [197]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [198]:
#Q1
df_og = df
df = df[base + target]
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [199]:
#Q2
df['horsepower'].median()

np.float64(254.0)

In [200]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]


In [201]:
#Q3
def prepare_X(_df, fill_value = 0):
    _df = _df.copy()
    _df.horsepower = _df.horsepower.fillna(fill_value)
    features = base
    X = _df[features].values
    X = np.column_stack([np.ones(X.shape[0]), X]) #fictional column of ones for w_0
    return X

In [202]:
#Q3 and Q4 regularization
def train(X,y, r = 0.0):
    XTX = X.T.dot(X)
    XTX += r * np.eye(XTX.shape[0])
    w = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w

In [203]:
def predict(X, w):
    return X.dot(w)

In [204]:
def RMSE(y_1, y_2):
    return np.sqrt(np.mean((y_1 - y_2)**2))

In [229]:
X_train_0 = prepare_X(df_train)
X_test_0 = prepare_X(df_test)
X_val_0 = prepare_X(df_val)

mean = df_train['horsepower'].mean()

X_train_mean = prepare_X(df_train, mean)
X_test_mean = prepare_X(df_test, mean)
X_val_mean = prepare_X(df_val, mean)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

w_0 = train(X_train_0, y_train)
w_mean = train(X_train_mean, y_train)

#print(RMSE(y_val, predict(X_val_0, w_0)))
#print(RMSE(y_val, predict(X_val_mean, w_mean)))


#Q4 Regularization
reg_list = [0, 0.01, 0.1, 1, 5, 10, 100]
min_er = RMSE(y_val, predict(X_val_0, w_0))
for r in reg_list:
    w = train(X_train_0, y_train, r)
    min_er = min(min_er, RMSE(y_val, predict(X_val_0, w)))
    print(f'r={r}; rmse={RMSE(y_val, predict(X_val_0, w))}\n')
print(f'Min RMSE is {min_er}')

r=0; rmse=2.207036592817896

r=0.01; rmse=2.2066564366671533

r=0.1; rmse=2.219526357279117

r=1; rmse=2.3364483047485587

r=5; rmse=2.39515163873769

r=10; rmse=2.4050615243783815

r=100; rmse=2.4146342231997986

Min RMSE is 2.2066564366671533


In [225]:
#Q5
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

rmse ={}

for seed in seeds:
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]

    X_train = prepare_X(df_train)
    X_val = prepare_X(df_val)
    X_test = prepare_X(df_test)

    y_train = df_train['fuel_efficiency_mpg'].values
    y_val = df_val['fuel_efficiency_mpg'].values
    y_test = df_test['fuel_efficiency_mpg'].values

    w_0 = train(X_train, y_train)

    rmse[f'seed_{seed}'] = RMSE(y_val, predict(X_val, w_0))

rounded_std_deviation = round(np.std(list(rmse.values())),3)
print(rounded_std_deviation)


0.029


In [230]:
#Q6
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

X_train = prepare_X(df_train)
X_val = prepare_X(df_val)
X_test = prepare_X(df_test)

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

r=0.001

X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

w = train(X_full, y_full, r)

RMSE(y_test, predict(X_test, w))

np.float64(2.235827747191731)